## **Notebook 11 — LangGraph RAG Pipeline**
- Wire the full RAG pipeline into a LangGraph StateGraph. Each retrieval and generation step becomes a node. Edges define the flow. A conditional edge adds a retry loop — if the answer has low confidence, the graph automatically retries with a broader search before giving up.

- Document used: `NLTK book` corpus (natural language processing textbook).
Why LangGraph over a plain function: A plain function runs linearly — no branching, no retries, no inspection. LangGraph gives you a visible graph with named nodes, typed state at every step, conditional routing, and built-in tracing. When something fails in production you can see exactly which node failed and what state it had.

## ***Part 1: Setup & Imports***

- LangGraph is a library built on top of LangChain that lets you define pipelines as directed graphs. StateGraph is the main class — you define a typed state, add nodes (functions), connect them with edges, and compile into a runnable chain.

- TypedDict for state: LangGraph passes one state object between every node. Using TypedDict makes every field explicit and type-checked. When you move to .py files you will replace this with a Pydantic model — the structure stays identical.

- Annotated + operator: Some fields use Annotated[list, operator.add]. This tells LangGraph how to merge state when nodes run in parallel — instead of overwriting the list, it appends to it. Used for the retrieved_chunks field since multiple retrieval nodes could contribute results

In [ ]:
import os, json, re, operator
from pathlib import Path
from typing import TypedDict, Annotated, Optional
from dataclasses import dataclass, field
from collections import Counter
from dotenv import load_dotenv

import numpy as np
import psycopg2
from pgvector.psycopg2 import register_vector
import bm25s

from FlagEmbedding import BGEM3FlagModel, FlagReranker
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# LangGraph core imports
from langgraph.graph import StateGraph, END

load_dotenv()

DATABASE_SYNC = os.getenv("DATABASE_URL_SYNC")
INDEX_DIR     = Path("../data/bm25_index_nltk")
INDEX_DIR.mkdir(parents=True, exist_ok=True)

print("All imports OK")
print(f"LangGraph imported successfully")

## **Cell 2 — define RAG state**
- The state is the most important design decision in LangGraph. Every node receives the full state as input and returns a partial state update as output. LangGraph merges the update into the current state before calling the next node.

**Field by field:**
- query: the original user question. Never modified after it is set.
- hyde_doc: the hypothetical answer generated by HyDE. Set by the transform node.
- hyde_embedding: the vector of the HyDE doc. Set by the transform node, used by dense search.
- retrieved_chunks: uses Annotated[list, operator.add] so parallel nodes append rather than overwrite.
- fused_chunks: RRF output — merged and re-ranked candidates.
- reranked_chunks: final top-5 after cross-encoder scoring.
- answer: the LLM-generated response.
- confidence: average reranker score of top-3 chunks. Used by the conditional edge to decide retry.
- retry_count: how many retries have happened. Prevents infinite loops.
- sources: citation metadata returned to the user.
- error: if any node fails, it writes here and the graph routes to END gracefully.

In [ ]:
class RAGState(TypedDict):
    """
    The single state object that flows through every node in the graph.
    Every node reads from this and writes partial updates back to it.
    LangGraph merges updates automatically between nodes.
    """
    # Input
    query           : str

    # Query transformation (HyDE node)
    hyde_doc        : Optional[str]
    hyde_embedding  : Optional[list[float]]

    # Retrieval nodes — Annotated[list, operator.add] means
    # if two nodes both return retrieved_chunks, the lists are
    # concatenated instead of one overwriting the other
    retrieved_chunks: Annotated[list, operator.add]

    # Fusion + reranking
    fused_chunks    : list
    reranked_chunks : list

    # Generation
    answer          : Optional[str]
    sources         : list

    # Control flow
    confidence      : float    # avg rerank score of top-3, used for retry decision
    retry_count     : int      # incremented on each retry, max=2
    error           : Optional[str]

print("RAGState defined")
print("Fields:", list(RAGState.__annotations__.keys()))

## **Cell 3 — load models + DB connection**
- **Why load models once at the top:** BGE-M3 is ~2.3 GB and takes 10-30 seconds to load. The reranker is ~1.1 GB. If you loaded them inside each node function, every query would take 30+ seconds just for model loading. Load once into module-level variables and reference them inside node functions.

- **Connection management note:** In this notebook we use a single persistent psycopg2 connection for simplicity. In the .py file with FastAPI + SQLModel you will use an async session factory with a connection pool — each request gets its own session, connections are returned to the pool after the request. The SQL queries stay identical.

In [ ]:
MODEL_DIR = Path("../models")

print("Loading BGE-M3...")
embed_model = BGEM3FlagModel(
    "BAAI/bge-m3", use_fp16=False,
    cache_dir=str(MODEL_DIR / "bge-m3")
)
print("  BGE-M3 ready")

print("Loading reranker...")
rerank_model = FlagReranker(
    "BAAI/bge-reranker-v2-m3", use_fp16=False,
    cache_dir=str(MODEL_DIR / "bge-reranker")
)
print("  Reranker ready")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)
print("  LLM ready")

# DB connection
conn_str = DATABASE_SYNC.replace("postgresql+psycopg2://", "postgresql://")
conn     = psycopg2.connect(conn_str)
register_vector(conn)
cur      = conn.cursor()
print("  DB connected")

print("\nAll components ready")

# ***PART 2 — Ingestion (NLTK Book)***

## **Cell 4 — download NLTK book + extract text**
- **Why NLTK book:** It is a freely available, well-structured text with clear chapters, sections, and technical content. Good for testing because questions have definite answers in the text.

- **What we do here:** Download the NLTK book corpus, extract the text from Chapter 1, save it as a plain text file in data/docs/. This gives us a real document to ingest without needing an external PDF.

- **Why Chapter 1 only:** The full NLTK book is very large. Chapter 1 alone has enough content (~50,000 words) to demonstrate the pipeline meaningfully without making embedding take 30+ minutes on CPU.

In [ ]:
import nltk

# Download the book corpus
nltk.download('book', quiet=True)
nltk.download('gutenberg', quiet=True)

from nltk.book import *

# Extract text from the first book — Moby Dick (large, structured)
# text1 is Moby Dick in the NLTK book corpus
moby_raw = " ".join(str(w) for w in text1.tokens[:50000])

# Also grab some NLTK intro text from the book module
intro_text = """
Chapter 1: Language Processing and Python

The Natural Language Toolkit (NLTK) is a platform for building Python programs
to work with human language data. It provides interfaces to over 50 corpora and
lexical resources, along with a suite of text processing libraries.

Tokenization is the process of breaking a stream of text up into words, phrases,
symbols, or other meaningful elements. Tokens can be individual words or sentences.

Frequency distribution shows how often each vocabulary item appears in the text.
NLTK provides FreqDist class to compute frequency distributions over words.

Collocations are sequences of words that appear together more often than expected
by chance. Common examples include red wine, hard work, and take a break.

Stop words are common words like the, is, at, which and on which are filtered out
before processing natural language data. NLTK provides a stopwords corpus.

Part-of-speech tagging assigns grammatical categories such as noun, verb, adjective
to each word in a sentence. NLTK provides pos_tag function for this purpose.

Named entity recognition identifies and classifies named entities in text into
categories such as person names, organizations, and locations.

Stemming reduces words to their root form by removing suffixes.
For example running becomes run and happiness becomes happi.

Lemmatization reduces words to their base form using vocabulary and morphological
analysis. Unlike stemming, lemmatization always returns a real dictionary word.

A corpus is a large collection of texts. The Brown Corpus was the first million-word
electronic corpus of English created at Brown University in 1961.

Bigrams are pairs of consecutive words in a text. Trigrams are sequences of three
consecutive words. N-grams are sequences of N consecutive words.

The Naive Bayes classifier is a probabilistic classifier based on applying Bayes
theorem with strong independence assumptions between features.

Regular expressions are sequences of characters that form a search pattern.
In NLTK, re module is used for regular expression operations on text.

Chunking is a process of extracting phrases from unstructured text using part of
speech tags. It is also called shallow parsing or light parsing.

WordNet is a large lexical database of English. Nouns, verbs, adjectives and adverbs
are grouped into sets of cognitive synonyms called synsets.

Concordance shows every occurrence of a given word together with some context.
NLTK concordance method takes a word and shows its uses in context.
"""

# Save to file
docs_dir = Path("../data/docs")
docs_dir.mkdir(parents=True, exist_ok=True)
nltk_path = docs_dir / "nltk_intro.txt"
nltk_path.write_text(intro_text, encoding="utf-8")

print(f"NLTK intro text saved: {nltk_path}")
print(f"Characters: {len(intro_text)}")
print(f"\nPreview:")
print(intro_text[:300])

## **Cell 5: Chunk insert into the DataBase**
- **Fresh table for this notebook:** We create chunks_nltk so this notebook is completely isolated from the BDRCS work.

- **Why these exact chunking parameters for NLTK text:** The NLTK intro text has shorter, denser sentences than an HR policy. We use child_size=2 (2 sentences per child) and parent_size=5 (5 sentences per parent) to match the content density. For HR policy we used 3 and 7. This is an important lesson — chunking parameters should be tuned per document type, not set once globally.

**The ingestion flow:** `parse → split sentences → build parent-child chunks → batch embed with BGE-M3 → insert into pgvector → build BM25s index.`
- All five steps happen here. Later when we build the .py files, each step becomes a separate function in a separate module.

In [ ]:
# ── Fresh table ──────────────────────────────────────────
cur.execute("DROP TABLE IF EXISTS chunks_nltk CASCADE;")
cur.execute("DROP TABLE IF EXISTS documents_nltk CASCADE;")

cur.execute("""
    CREATE TABLE documents_nltk (
        id        UUID PRIMARY KEY DEFAULT gen_random_uuid(),
        filename  TEXT NOT NULL,
        status    TEXT DEFAULT 'pending',
        created_at TIMESTAMPTZ DEFAULT now()
    );
""")
cur.execute("""
    CREATE TABLE chunks_nltk (
        id             UUID PRIMARY KEY DEFAULT gen_random_uuid(),
        document_id    UUID REFERENCES documents_nltk(id) ON DELETE CASCADE,
        content        TEXT NOT NULL,
        parent_content TEXT,
        embedding      vector(1024),
        metadata       JSONB NOT NULL DEFAULT '{}',
        created_at     TIMESTAMPTZ DEFAULT now()
    );
""")
cur.execute("""
    CREATE INDEX idx_nltk_embedding ON chunks_nltk
    USING hnsw (embedding vector_cosine_ops)
    WITH (m=16, ef_construction=64);
""")
cur.execute("""
    CREATE INDEX idx_nltk_metadata ON chunks_nltk USING gin(metadata);
""")
conn.commit()
print("Tables created: documents_nltk, chunks_nltk")

# ── Chunking helpers ──────────────────────────────────────
def split_sentences(text: str) -> list[str]:
    sentences = []
    for line in text.split("\n"):
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = re.split(r'(?<=[.!?])\s+', line)
        for part in parts:
            if len(part.strip()) > 20:
                sentences.append(part.strip())
    return sentences

def detect_section(lines: list[str], idx: int) -> str:
    for line in reversed(lines[:idx]):
        line = line.strip()
        if line.startswith("Chapter") or line.startswith("##"):
            return line[:60]
    return "General"

def build_chunks(text, filename, child_size=2, parent_size=5, overlap=1):
    sentences = split_sentences(text)
    lines     = text.split("\n")
    chunks, idx, i = [], 0, 0
    while i < len(sentences):
        parent_sents   = sentences[i:i+parent_size]
        parent_content = " ".join(parent_sents)
        section        = detect_section(lines, min(i*2, len(lines)-1))
        j = 0
        while j < len(parent_sents):
            child_content = " ".join(parent_sents[j:j+child_size])
            if len(child_content.strip()) > 20:
                chunks.append({
                    "content"       : child_content,
                    "parent_content": parent_content,
                    "section"       : section,
                    "metadata"      : {
                        "filename": filename,
                        "section" : section,
                        "chunk_idx": idx,
                    }
                })
                idx += 1
            j += child_size - overlap
        i += parent_size - overlap
    return chunks

# ── Ingest NLTK text ──────────────────────────────────────
text     = nltk_path.read_text(encoding="utf-8")
chunks   = build_chunks(text, "nltk_intro.txt")

print(f"\nChunks built: {len(chunks)}")

cur.execute("INSERT INTO documents_nltk (filename, status) VALUES (%s,'ready') RETURNING id;",
            ("nltk_intro.txt",))
doc_id = cur.fetchone()[0]

# Batch embed
texts    = [c["content"] for c in chunks]
all_vecs = []
for i in range(0, len(texts), 16):
    batch = texts[i:i+16]
    vecs  = embed_model.encode(batch, return_dense=True,
                               return_sparse=False)["dense_vecs"]
    all_vecs.extend(vecs)

for chunk, vec in zip(chunks, all_vecs):
    cur.execute("""
        INSERT INTO chunks_nltk
            (document_id, content, parent_content, embedding, metadata)
        VALUES (%s,%s,%s,%s,%s::jsonb)
    """, (doc_id, chunk["content"], chunk["parent_content"],
          vec.tolist(), json.dumps(chunk["metadata"])))

conn.commit()
print(f"Stored {len(chunks)} chunks in pgvector")

# Build BM25s index
corpus_tokens  = bm25s.tokenize(texts, stopwords="en")
bm25_ret       = bm25s.BM25()
bm25_ret.index(corpus_tokens)
bm25_ret.save(str(INDEX_DIR / "bm25_index"))

cur.execute("SELECT id::text FROM chunks_nltk ORDER BY created_at;")
chunk_ids = [r[0] for r in cur.fetchall()]

with open(INDEX_DIR/"chunk_ids.json",  "w") as f: json.dump(chunk_ids, f)
with open(INDEX_DIR/"chunk_texts.json","w") as f: json.dump(texts,     f)

print(f"BM25s index saved with {len(chunk_ids)} entries")
print("\nIngestion complete — ready to build the graph")

# **── Global variables used by retrieval nodes ──────────────**
-  These must exist in the notebook scope before nodes are defined.
-  If you restart the kernel, re-run Cell 5 first to repopulate these.

In [ ]:


with open(INDEX_DIR / "chunk_ids.json",   "r") as f:
    chunk_ids   = json.load(f)

with open(INDEX_DIR / "chunk_texts.json", "r") as f:
    chunk_texts = json.load(f)

# Reload BM25s index from disk
bm25_ret = bm25s.BM25.load(
    str(INDEX_DIR / "bm25_index"),
    load_corpus=False
)

print(f"chunk_ids   : {len(chunk_ids)} entries")
print(f"chunk_texts : {len(chunk_texts)} entries")
print(f"bm25_ret    : loaded OK")

# ***PART 3 — LangGraph Node Functions***

## **Cell 6 — node 1: transform_query**
- **What a node is:** A node is just a Python function. It receives the full RAGState and returns a dictionary of fields to update. LangGraph merges the returned dict into the current state. You never return the full state — only the fields that changed.

- **What this node does:** Implements HyDE (Hypothetical Document Embedding). It asks the LLM to write a fake answer to the query, then embeds that fake answer. The resulting vector is stored in hyde_embedding for the dense search node to use.

- **Why this is a separate node:** Separating it means you can skip it (connect dense directly to query embedding), replace it (swap HyDE for multi-query expansion), or retry it independently. In LangGraph, every node is optional — you wire only what you need.

In [ ]:
HYDE_PROMPT = """You are an NLP and Python programming expert.
Write a short passage (3-5 sentences) that directly answers the question below.
Write as if the passage was extracted from an official NLP textbook.
Use formal technical language. Do NOT say "I". Just write the passage directly.

Question: {query}
Passage:"""

def transform_query(state: RAGState) -> dict:
    """
    NODE 1 — Query Transformation (HyDE)

    Input state fields used : query
    Output state fields set : hyde_doc, hyde_embedding

    Generates a hypothetical answer to the query using the LLM.
    Embeds the hypothetical answer with BGE-M3.
    The embedding is used by the dense search node instead of
    embedding the raw query — this bridges vocabulary mismatch.

    Example:
      Query   : "What is tokenization in NLTK?"
      HyDE doc: "Tokenization is the process of splitting text into
                 individual words or sentences. NLTK provides word_tokenize
                 and sent_tokenize functions for this purpose."
      The HyDE doc embedding matches the indexed text far better
      than the raw query embedding would.
    """
    print(f"  [transform_query] Generating HyDE doc for: '{state['query'][:50]}'")

    try:
        response = llm.invoke([
            HumanMessage(content=HYDE_PROMPT.format(query=state["query"]))
        ])
        hyde_doc = response.content

        hyde_vec = embed_model.encode(
            [hyde_doc], return_dense=True
        )["dense_vecs"][0].tolist()

        print(f"  [transform_query] HyDE doc: '{hyde_doc[:80]}...'")

        return {
            "hyde_doc"       : hyde_doc,
            "hyde_embedding" : hyde_vec,
        }

    except Exception as e:
        print(f"  [transform_query] ERROR: {e}")
        # Fallback — embed raw query if HyDE fails
        raw_vec = embed_model.encode(
            [state["query"]], return_dense=True
        )["dense_vecs"][0].tolist()
        return {
            "hyde_doc"       : state["query"],
            "hyde_embedding" : raw_vec,
            "error"          : f"HyDE failed, using raw query: {e}",
        }

print("transform_query node defined")

## **Cell 7 — node 2: retrieve_dense**
- **What this node does:** Runs cosine similarity search against pgvector using the HyDE embedding from the previous node. Returns the top-20 semantically similar chunks.

**Why it reads hyde_embedding not query:** The HyDE embedding is a better query vector than embedding the raw question. It uses the same vocabulary as the documents. But we keep the raw query available in state for the sparse search node — BM25s works on raw text, not vectors.

**The retrieved_chunks append pattern:** This node returns {"retrieved_chunks": dense_results}. Because retrieved_chunks is annotated with operator.add, LangGraph appends this list to whatever is already in retrieved_chunks — it does not overwrite. This means when the sparse node also returns results, both lists are combined automatically.

In [ ]:
def retrieve_dense(state: RAGState) -> dict:
    """
    NODE 2 — Dense Retrieval (pgvector)

    Input state fields used : hyde_embedding
    Output state fields set : retrieved_chunks (appended, not replaced)

    Runs cosine similarity search in pgvector using the HyDE embedding.
    Returns top-20 results. These are the semantically relevant chunks —
    they match meaning even if they use different words than the query.

    The results are tagged with source='dense' so the fusion node
    can tell which retriever produced each result.
    """
    print(f"  [retrieve_dense] Running pgvector search...")

    try:
        query_vec = state["hyde_embedding"]

        cur.execute("""
            SELECT
                id::text,
                content,
                parent_content,
                metadata->>'section',
                1 - (embedding <=> %s::vector) AS similarity
            FROM chunks_nltk
            ORDER BY embedding <=> %s::vector
            LIMIT 20;
        """, (query_vec, query_vec))

        results = [
            {
                "chunk_id"       : r[0],
                "content"        : r[1],
                "parent_content" : r[2],
                "section"        : r[3],
                "score"          : float(r[4]),
                "rank"           : i + 1,
                "source"         : "dense",
            }
            for i, r in enumerate(cur.fetchall())
        ]

        print(f"  [retrieve_dense] Found {len(results)} chunks")
        return {"retrieved_chunks": results}

    except Exception as e:
        print(f"  [retrieve_dense] ERROR: {e}")
        return {"retrieved_chunks": [], "error": str(e)}

print("retrieve_dense node defined")

## **Cell 8 — node 3: retrieve_sparse**
- **What this node does:** Runs BM25s keyword search using the raw query text. Returns the top-20 keyword-matched chunks.

- **Why raw query, not HyDE doc:** BM25s is a keyword matching algorithm — it counts exact token occurrences. The raw user query contains the exact keywords the user typed, which are often the best signal for BM25s. The HyDE doc adds extra words that might dilute the keyword signal.

In [ ]:
def retrieve_sparse(state: RAGState) -> dict:
    """
    NODE 3 — Sparse Retrieval (BM25s)

    Input state fields used : query
    Output state fields set : retrieved_chunks (appended to dense results)

    Runs BM25 keyword search against the in-memory index.
    Uses raw query text — BM25s works on exact tokens.
    Returns top-20 keyword-matched chunks.

    Complements dense search:
    - Dense finds 'tokenization' when query says 'splitting text into words'
    - Sparse finds 'word_tokenize' when query says 'word_tokenize function'
    """
    print(f"  [retrieve_sparse] Running BM25s search...")

    try:
        query_tokens    = bm25s.tokenize([state["query"]], stopwords="en")
        results, scores = bm25_ret.retrieve(
            query_tokens, k=min(20, len(chunk_ids))
        )

        sparse_results = [
            {
                "chunk_id"       : chunk_ids[idx],
                "content"        : chunk_texts[idx],
                "parent_content" : None,
                "section"        : None,
                "score"          : float(score),
                "rank"           : i + 1,
                "source"         : "sparse",
            }
            for i, (idx, score) in enumerate(zip(results[0], scores[0]))
        ]

        print(f"  [retrieve_sparse] Found {len(sparse_results)} chunks")
        return {"retrieved_chunks": sparse_results}

    except Exception as e:
        print(f"  [retrieve_sparse] ERROR: {e}")
        return {"retrieved_chunks": [], "error": str(e)}

print("retrieve_sparse node defined")

## **Cell 9 — node 4: fuse_results**
- **What this node does:** Takes all chunks in retrieved_chunks (which now contains both dense and sparse results from the previous two nodes), splits them by source tag, and runs RRF fusion.

- **Why this is a separate node:** Fusion is a distinct logical step. Keeping it separate means you can inspect the state after retrieval and before fusion — you can see exactly what each retriever returned before they are merged. This is invaluable for debugging retrieval quality.

- **Enrichment step:** Sparse results do not carry parent_content or section because the BM25s index does not store them. After fusion, we query pgvector to fill in the missing fields for any chunk that came from sparse-only. This ensures the generation node always has full context regardless of which retriever found each chunk.

In [ ]:
def fuse_results(state: RAGState) -> dict:
    """
    NODE 4 — RRF Fusion

    Input state fields used : retrieved_chunks (combined dense + sparse)
    Output state fields set : fused_chunks

    Splits retrieved_chunks back into dense and sparse lists.
    Applies Reciprocal Rank Fusion (k=60).
    Enriches sparse-only results with parent_content from pgvector.
    Returns top-10 fused candidates for the reranker.

    RRF formula: score(chunk) = sum of 1/(60 + rank) across retrievers
    Chunks in both lists score ~2x higher than chunks in one list.
    Scale-independent — BM25 scores (0-20) and cosine scores (0-1) mix safely.
    """
    print(f"  [fuse_results] Fusing {len(state['retrieved_chunks'])} total candidates...")

    all_chunks    = state["retrieved_chunks"]
    dense_results = [c for c in all_chunks if c.get("source") == "dense"]
    sparse_results= [c for c in all_chunks if c.get("source") == "sparse"]

    print(f"  [fuse_results] Dense: {len(dense_results)}, Sparse: {len(sparse_results)}")

    # RRF scoring
    rrf_scores: dict[str, float] = {}
    for r in dense_results:
        cid = r["chunk_id"]
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0 / (60 + r["rank"])
    for r in sparse_results:
        cid = r["chunk_id"]
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0 / (60 + r["rank"])

    ranked   = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:10]
    d_lookup = {r["chunk_id"]: r for r in dense_results}

    fused = []
    for chunk_id, rrf_score in ranked:
        base = d_lookup.get(chunk_id, {})
        fused.append({
            "chunk_id"       : chunk_id,
            "content"        : base.get("content", ""),
            "parent_content" : base.get("parent_content"),
            "section"        : base.get("section"),
            "rrf_score"      : rrf_score,
            "rank"           : len(fused) + 1,
            "in_dense"       : chunk_id in {r["chunk_id"] for r in dense_results},
            "in_sparse"      : chunk_id in {r["chunk_id"] for r in sparse_results},
        })

    # Enrich sparse-only chunks with parent_content + section from DB
    missing = [r for r in fused if not r.get("parent_content")]
    if missing:
        placeholders = ",".join(["%s"] * len(missing))
        cur.execute(
            f"SELECT id::text, content, parent_content, metadata->>'section' "
            f"FROM chunks_nltk WHERE id::text IN ({placeholders});",
            [r["chunk_id"] for r in missing]
        )
        enriched = {r[0]: r for r in cur.fetchall()}
        for r in fused:
            if r["chunk_id"] in enriched:
                e = enriched[r["chunk_id"]]
                r["content"]        = r["content"]        or e[1]
                r["parent_content"] = r["parent_content"] or e[2]
                r["section"]        = r["section"]        or e[3]

    print(f"  [fuse_results] Fused to {len(fused)} candidates")
    return {"fused_chunks": fused}

print("fuse_results node defined")

## **Cell 10 — node 5: rerank_chunks**

- **What this node does:** Takes the top-10 fused candidates and scores each (query, chunk) pair with the cross-encoder. Re-sorts by reranker score. Sets the confidence field — the average reranker score of the top-3 results.

- **Why confidence matters:** The confidence score is used by the conditional edge after this node. If confidence is below 0.4, it means the reranker found no chunks that are clearly relevant to the query. The graph will then retry with a broader search (relaxed parameters) instead of generating a low-quality answer. If confidence is above 0.4, generation proceeds normally.

- **The cross-encoder difference:** BGE-M3 (bi-encoder) embeds query and document separately then computes cosine similarity — fast but approximate. BGE-reranker (cross-encoder) concatenates query + document as a single input — the model attends to both simultaneously. This joint attention catches subtle relevance signals that cosine similarity misses entirely.

In [ ]:
def rerank_chunks(state: RAGState) -> dict:
    """
    NODE 5 — Cross-Encoder Reranking

    Input state fields used : query, fused_chunks
    Output state fields set : reranked_chunks, confidence

    Scores each (query, chunk) pair jointly with the cross-encoder.
    Much more accurate than cosine similarity — reads both together.
    Only practical on small candidate sets (10-20 docs).

    Sets confidence = avg rerank score of top-3 results.
    Confidence < 0.4 → conditional edge routes to retry
    Confidence >= 0.4 → conditional edge routes to generation
    """
    print(f"  [rerank_chunks] Reranking {len(state['fused_chunks'])} candidates...")

    candidates = state["fused_chunks"]
    if not candidates:
        return {"reranked_chunks": [], "confidence": 0.0}

    pairs  = [[state["query"], c["content"]] for c in candidates]
    scores = rerank_model.compute_score(pairs, normalize=True)

    for candidate, score in zip(candidates, scores):
        candidate["rerank_score"] = float(score)

    reranked = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)
    for i, r in enumerate(reranked): r["rank"] = i + 1

    top5       = reranked[:5]
    top3_scores= [r["rerank_score"] for r in reranked[:3]]
    confidence = float(np.mean(top3_scores)) if top3_scores else 0.0

    print(f'  [rerank_chunks] Top scores: {[f"{r["rerank_score"]:.3f}" for r in top5]}')
    print(f"  [rerank_chunks] Confidence: {confidence:.3f}")

    return {
        "reranked_chunks": top5,
        "confidence"     : confidence,
    }

print("rerank_chunks node defined")

## **Cell 11 — node 6: generate_answer**

- **What this node does:** Builds the grounded RAG prompt from the top-5 reranked chunks and calls GPT-4o-mini. Uses parent_content (larger context window) not content (small child chunk) so the LLM has enough surrounding text to answer correctly.

- **The citation mechanism:** Each chunk is labelled [1], [2], [3] etc. in the prompt. The system prompt instructs the LLM to cite every claim using these labels. After generation we build the sources list that maps each [N] label back to the actual section name and content preview — this is what gets returned to the user alongside the answer.

- **Temperature=0.1 for RAG:** Low temperature means near-deterministic output. The LLM should follow the context faithfully and not be creative. Creative generation is the enemy of grounded RAG — it leads to hallucination.

In [ ]:
SYSTEM_PROMPT = """You are a helpful NLP and Python programming assistant.
Answer questions ONLY using the context passages provided.
If the context does not contain sufficient information, say:
"I could not find a clear answer in the provided documents."

Rules:
- Cite every factual claim using passage numbers like [1] or [2][3]
- Be concise and direct
- Never add information not present in the context
- Never use your own knowledge — context passages only"""

def generate_answer(state: RAGState) -> dict:
    """
    NODE 6 — LLM Generation

    Input state fields used : query, reranked_chunks
    Output state fields set : answer, sources

    Builds a numbered context prompt from the top-5 reranked chunks.
    Uses parent_content for richer context around each retrieved sentence.
    Calls GPT-4o-mini with strict grounding instructions.
    Parses citations and builds source metadata for the user.

    This node only runs if confidence >= 0.4 (conditional edge).
    If confidence is low the retry node runs first.
    """
    print(f"  [generate_answer] Generating answer...")

    chunks = state["reranked_chunks"]
    if not chunks:
        return {
            "answer" : "I could not find relevant information for your query.",
            "sources": [],
        }

    # Build numbered context — use parent for richer context
    parts = []
    for i, chunk in enumerate(chunks, 1):
        text    = chunk.get("parent_content") or chunk.get("content", "")
        section = chunk.get("section", "Unknown")
        parts.append(f"[{i}] (Section: {section})\n{text}")

    context  = "\n\n".join(parts)
    prompt   = f"""CONTEXT PASSAGES:
{context}

QUESTION: {state['query']}

ANSWER (cite using [1], [2] etc):"""

    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=prompt),
    ])

    sources = [
        {
            "citation"     : f"[{i}]",
            "section"      : c.get("section", ""),
            "preview"      : (c.get("content") or "")[:80],
            "rerank_score" : c.get("rerank_score", 0),
        }
        for i, c in enumerate(chunks, 1)
    ]

    print(f"  [generate_answer] Answer generated ({len(response.content)} chars)")

    return {
        "answer" : response.content,
        "sources": sources,
    }

print("generate_answer node defined")

## **Cell 12 — node 7: retry_retrieval**

- **What this node does:** Handles the case where confidence is too low. It re-runs retrieval with relaxed parameters — fetching more candidates (k=30 instead of k=20) and broadening the BM25s search. It also resets retrieved_chunks to an empty list so the next run starts fresh rather than appending to stale results. Increments retry_count to prevent infinite loops.

- **Why not just fail immediately:** Low confidence often means the query phrasing did not match the indexed text well. A second attempt with relaxed parameters frequently finds the answer. This is especially true for technical queries where the user uses informal language and the document uses formal language.

- **Max 2 retries:** After 2 failed attempts we route to generation anyway and let the LLM admit it could not find a clear answer. Infinite retry loops are worse than an honest "I don't know."

In [ ]:
def retry_retrieval(state: RAGState) -> dict:
    """
    NODE 7 — Retry Retrieval (low confidence path)

    Input state fields used : query, retry_count
    Output state fields set : retrieved_chunks (reset), fused_chunks,
                              reranked_chunks, confidence, retry_count

    Called when confidence < 0.4 AND retry_count < 2.
    Re-runs full retrieval with broader parameters (k=30).
    Resets retrieved_chunks to [] so the operator.add annotation
    does not keep appending to the previous failed attempt.

    After retry, routes back to rerank_chunks via conditional edge.
    Max 2 retries — after that, routes to generate_answer regardless.
    """
    retry_num = state.get("retry_count", 0) + 1
    print(f"  [retry_retrieval] Retry #{retry_num} — broadening search...")

    # ── Dense with raw query embedding (not HyDE) ────────────
    raw_vec = embed_model.encode(
        [state["query"]], return_dense=True
    )["dense_vecs"][0].tolist()

    cur.execute("""
        SELECT id::text, content, parent_content,
               metadata->>'section', 1-(embedding <=> %s::vector)
        FROM chunks_nltk ORDER BY embedding <=> %s::vector LIMIT 30;
    """, (raw_vec, raw_vec))

    dense = [{"chunk_id":r[0],"content":r[1],"parent_content":r[2],
              "section":r[3],"score":float(r[4]),"rank":i+1,"source":"dense"}
             for i,r in enumerate(cur.fetchall())]

    # ── Sparse with broader k ─────────────────────────────────
    tokens = bm25s.tokenize([state["query"]], stopwords="en")
    res, sc = bm25_ret.retrieve(tokens, k=min(30, len(chunk_ids)))
    sparse  = [{"chunk_id":chunk_ids[idx],"content":chunk_texts[idx],
                "parent_content":None,"section":None,
                "score":float(s),"rank":i+1,"source":"sparse"}
               for i,(idx,s) in enumerate(zip(res[0],sc[0]))]

    # ── RRF fusion ────────────────────────────────────────────
    scores = {}
    for r in dense:
        scores[r["chunk_id"]] = scores.get(r["chunk_id"],0)+1/(60+r["rank"])
    for r in sparse:
        scores[r["chunk_id"]] = scores.get(r["chunk_id"],0)+1/(60+r["rank"])

    ranked   = sorted(scores.items(), key=lambda x:x[1], reverse=True)[:10]
    d_lookup = {r["chunk_id"]:r for r in dense}
    fused    = []
    for cid, rrf_score in ranked:
        base = d_lookup.get(cid,{})
        fused.append({"chunk_id":cid,"content":base.get("content",""),
                      "parent_content":base.get("parent_content"),
                      "section":base.get("section"),
                      "rrf_score":rrf_score,"rank":len(fused)+1,
                      "in_dense":cid in {r["chunk_id"] for r in dense},
                      "in_sparse":cid in {r["chunk_id"] for r in sparse}})

    # ── Rerank ────────────────────────────────────────────────
    pairs  = [[state["query"], c["content"]] for c in fused]
    sc_r   = rerank_model.compute_score(pairs, normalize=True)
    for c,s in zip(fused,sc_r): c["rerank_score"]=float(s)
    reranked = sorted(fused, key=lambda x:x["rerank_score"], reverse=True)[:5]
    for i,r in enumerate(reranked): r["rank"]=i+1

    confidence = float(np.mean([r["rerank_score"] for r in reranked[:3]])) \
                 if reranked else 0.0

    print(f"  [retry_retrieval] New confidence: {confidence:.3f}")

    return {
        "retrieved_chunks": [],   # reset — prevents appending to stale list
        "fused_chunks"    : fused,
        "reranked_chunks" : reranked,
        "confidence"      : confidence,
        "retry_count"     : retry_num,
    }

print("retry_retrieval node defined")

# ***PART 4 — Conditional Edge + Graph Assembly***

<hr>

## **Cell 13 — conditional edge function**

- **What a conditional edge is:** In LangGraph, a conditional edge is a function that reads the current state and returns a string — the name of the next node to route to. This is how branching works. After the reranker runs, instead of always going to generation, we call this function to decide: generate, retry, or end with error.

**The three routes:**
- "generate": confidence >= 0.4, everything is fine, proceed to answer generation.
- "retry": confidence < 0.4 AND retry_count < 2, we have not tried enough times yet, go back and try again with broader parameters.
- "end_low_confidence": confidence < 0.4 AND retry_count >= 2, we have retried twice and still found nothing relevant, route to a graceful end node that returns a polite "I don't know."

- **The error route:** If any node set the error field, we route to end immediately regardless of confidence. This prevents the graph from running generation on corrupted state.

In [ ]:
# def should_generate_or_retry(state: RAGState) -> str:
#     """
#     CONDITIONAL EDGE — routes after rerank_chunks

#     Reads confidence and retry_count from state.
#     Returns the name of the next node to execute.

#     Decision logic:
#     1. If error set by any node → go to end_with_error
#     2. If confidence >= 0.4    → go to generate_answer (good retrieval)
#     3. If confidence < 0.4 AND retry_count < 2 → go to retry_retrieval
#     4. If confidence < 0.4 AND retry_count >= 2 → go to end_low_confidence

#     Confidence threshold 0.4:
#     - Above 0.4 means at least some chunks are reasonably relevant
#     - Below 0.4 means even the best candidates are poor matches
#     - This value can be tuned per domain — 0.4 works for general use
#     """
#     confidence  = state.get("confidence", 0.0)
#     retry_count = state.get("retry_count", 0)
#     error       = state.get("error")

#     print(f"\n  [edge] confidence={confidence:.3f}  retry={retry_count}  error={bool(error)}")

#     if error and "HyDE failed" not in str(error):
#         print(f"  [edge] → end_with_error")
#         return "end_with_error"

#     if confidence >= 0.4:
#         print(f"  [edge] → generate_answer")
#         return "generate"

#     if retry_count < 2:
#         print(f"  [edge] → retry_retrieval")
#         return "retry"

#     print(f"  [edge] → end_low_confidence (gave up after {retry_count} retries)")
#     return "end_low_confidence"

# print("Conditional edge function defined")


def should_generate_or_retry(state: RAGState) -> str:
    """
    CONDITIONAL EDGE — routes after rerank_chunks

    Reads confidence and retry_count from state.
    Returns the name of the next node to execute.

    Decision logic:
    1. If error set by any node → go to end_with_error
    2. If confidence >= 0.4    → go to generate_answer (good retrieval)
    3. If confidence < 0.4 AND retry_count < 2 → go to retry_retrieval
    4. If confidence < 0.4 AND retry_count >= 2 → go to end_low_confidence

    Confidence threshold 0.4:
    - Above 0.4 means at least some chunks are reasonably relevant
    - Below 0.4 means even the best candidates are poor matches
    - This value can be tuned per domain — 0.4 works for general use
    """

    confidence  = state.get("confidence", 0.0)
    retry_count = state.get("retry_count", 0)
    error       = state.get("error")

    print(f"\n  [edge] confidence={confidence:.3f}  "
          f"retry={retry_count}  error={bool(error)}")

    # Only hard-fail on critical errors — not retrieval warnings
    # Sparse search failure is recoverable (dense alone is enough)
    critical_errors = ["database", "connection", "pgvector", "SSL"]
    if error and any(e.lower() in str(error).lower() for e in critical_errors):
        print(f"  [edge] → end_with_error (critical: {error})")
        return "end_with_error"

    # Normal routing based on confidence
    if confidence >= 0.4:
        print(f"  [edge] → generate_answer")
        return "generate"

    if retry_count < 2:
        print(f"  [edge] → retry_retrieval")
        return "retry"

    print(f"  [edge] → end_low_confidence")
    return "end_low_confidence"

## **Cell 14 — terminal nodes (graceful endings)**

- **Why terminal nodes:** LangGraph requires every path through the graph to eventually reach END. When the confidence is permanently low or an error occurred, we need a node that writes a helpful message to answer and then routes to END. Without these, the graph would have dangling edges and fail to compile.

- **Graceful degradation principle:** A RAG system should never crash or return an empty response. If retrieval fails, the user should get a polite, honest message. These terminal nodes implement that guarantee.

In [ ]:
def end_low_confidence(state: RAGState) -> dict:
    """
    TERMINAL NODE — low confidence path

    Called when confidence < 0.4 after 2 retries.
    Returns an honest "I don't know" message with the query echoed back.
    This is better than generating a hallucinated answer.
    """
    print(f"  [end_low_confidence] Could not find relevant content after retries")
    return {
        "answer" : (
            f"I could not find relevant information in the documents "
            f"to answer: '{state['query']}'. "
            f"Please try rephrasing your question or check if the topic "
            f"is covered in the indexed documents."
        ),
        "sources": [],
    }

def end_with_error(state: RAGState) -> dict:
    """
    TERMINAL NODE — error path

    Called when a critical error occurred in any node.
    Returns the error message so the caller can debug.
    """
    print(f"  [end_with_error] Pipeline error: {state.get('error')}")
    return {
        "answer" : f"Pipeline error: {state.get('error', 'Unknown error')}",
        "sources": [],
    }

print("Terminal nodes defined")

## **Cell 15 — build and compile the graph**

`How StateGraph is assembled`
1. Create a StateGraph with your state type.
2. Add every node with add_node(name, function).
3. Set the entry point — the first node to execute.
4. Add edges — add_edge(from, to) for unconditional flow, add_conditional_edges(from, fn, mapping) for branching.
5. Call compile() — this validates the graph, checks all paths reach END, and returns a runnable object.

**The graph structure:**
- `START → transform_query → retrieve_dense → retrieve_sparse → fuse_results → rerank_chunks → [conditional edge] → generate_answer → END`
- **With retry loop:** `rerank_chunks → retry_retrieval → rerank_chunks → ... (max 2 times)`

- **Why compile() matters:** Compilation validates the graph structure. If you have a node with no outgoing edge, a missing node reference, or a state field type mismatch, compilation fails with a clear error. This catches mistakes before you run the first query.

In [ ]:
# ── Build the graph ───────────────────────────────────────
builder = StateGraph(RAGState)

# Add all nodes
builder.add_node("transform_query",    transform_query)
builder.add_node("retrieve_dense",     retrieve_dense)
builder.add_node("retrieve_sparse",    retrieve_sparse)
builder.add_node("fuse_results",       fuse_results)
builder.add_node("rerank_chunks",      rerank_chunks)
builder.add_node("generate_answer",    generate_answer)
builder.add_node("retry_retrieval",    retry_retrieval)
builder.add_node("end_low_confidence", end_low_confidence)
builder.add_node("end_with_error",     end_with_error)

# Entry point
builder.set_entry_point("transform_query")

# Linear flow: transform → dense → sparse → fuse → rerank
builder.add_edge("transform_query", "retrieve_dense")
builder.add_edge("retrieve_dense",  "retrieve_sparse")
builder.add_edge("retrieve_sparse", "fuse_results")
builder.add_edge("fuse_results",    "rerank_chunks")

# Conditional edge after reranking — branches to 4 possible next nodes
builder.add_conditional_edges(
    "rerank_chunks",              # source node
    should_generate_or_retry,     # routing function
    {                             # mapping: return value → next node
        "generate"          : "generate_answer",
        "retry"             : "retry_retrieval",
        "end_low_confidence": "end_low_confidence",
        "end_with_error"    : "end_with_error",
    }
)

# After retry → go back to rerank (retry already ran fusion + rerank internally)
# We route to generate_answer directly since retry sets reranked_chunks itself
builder.add_conditional_edges(
    "retry_retrieval",
    should_generate_or_retry,
    {
        "generate"          : "generate_answer",
        "retry"             : "retry_retrieval",
        "end_low_confidence": "end_low_confidence",
        "end_with_error"    : "end_with_error",
    }
)

# Terminal edges
builder.add_edge("generate_answer",    END)
builder.add_edge("end_low_confidence", END)
builder.add_edge("end_with_error",     END)

# Compile — validates graph structure
rag_graph = builder.compile()

print("Graph compiled successfully")
print()
print("Graph nodes:")
for node in builder.nodes:
    print(f"  {node}")
print()

print("Graph edges defined — ready to run")

In [ ]:
rag_graph

## **Cell 16 — helper to run the graph + print trace**
- **Initial state:** When you invoke a LangGraph graph you pass the initial state. Only set the fields you know at the start — everything else gets filled by nodes as the graph runs. We set query, retrieved_chunks=[] (required because it uses operator.add), and the control flow counters at their defaults.

- **The trace:** LangGraph returns the final state after all nodes have executed. By printing intermediate fields like hyde_doc, confidence, and retry_count we can see exactly what happened at each stage — this is the observability advantage over a plain function pipeline.

In [ ]:
def run_rag(query: str) -> dict:
    """
    Run the compiled LangGraph RAG pipeline.

    Initialises state with the query and default values.
    Invokes the graph — nodes execute in sequence following edges.
    Returns the final state with answer, sources, and trace info.
    """
    initial_state: RAGState = {
        "query"           : query,
        "hyde_doc"        : None,
        "hyde_embedding"  : None,
        "retrieved_chunks": [],     # must be [] for operator.add to work
        "fused_chunks"    : [],
        "reranked_chunks" : [],
        "answer"          : None,
        "sources"         : [],
        "confidence"      : 0.0,
        "retry_count"     : 0,
        "error"           : None,
    }

    print(f"\n{'='*60}")
    print(f"QUERY: {query}")
    print(f"{'='*60}")

    final_state = rag_graph.invoke(initial_state)

    return final_state

def print_result(state: dict):
    """Print final state in a readable format."""
    print(f"\nANSWER:")
    print(f"  {state['answer']}")
    print(f"\nSOURCES:")
    for s in state.get("sources", []):
        print(f"  {s['citation']} [{s['section']}]  "
              f"score:{s['rerank_score']:.3f}")
        print(f"       {s['preview']}")
    print(f"\nPIPELINE TRACE:")
    print(f"  HyDE doc preview : {(state.get('hyde_doc') or '')[:80]}")
    print(f"  Confidence       : {state.get('confidence', 0):.3f}")
    print(f"  Retry count      : {state.get('retry_count', 0)}")
    print(f"  Chunks retrieved : {len(state.get('retrieved_chunks', []))}")
    print(f"  Chunks fused     : {len(state.get('fused_chunks', []))}")
    print(f"  Chunks reranked  : {len(state.get('reranked_chunks', []))}")

print("run_rag() and print_result() ready")

## **Cell 17 — test query 1: factual NLP question**
- **Tests the normal happy path:** A clear factual question about NLTK that the document covers directly. Expect confidence >= 0.4, no retry, answer with citations. This verifies the main flow works end to end.


In [ ]:
state1 = run_rag("What is tokenization in NLTK and how does it work?")
print_result(state1)

In [ ]:
state2 = run_rag("What does FreqDist do in NLTK?")
print_result(state2)

## **Cell 19 — test query 3: out of scope (triggers low confidence path)**

- **Tests the conditional retry + graceful end path:** A question completely outside the indexed document. Confidence should be low, the graph retries once or twice, then routes to end_low_confidence which returns an honest "I don't know." This verifies the safety mechanism works — the pipeline does not hallucinate when it cannot find relevant content.

In [ ]:
state3 = run_rag("What is the population of Bangladesh in 2024?")
print_result(state3)

## **Cell 20 — test query 4: semantic paraphrase**
- **Tests HyDE + dense contribution:** A query that uses completely different words from the document ("base form of a word" vs "lemmatization"). Dense search with HyDE embedding should find the correct chunk. BM25s would miss this. The pipeline trace will show the answer coming from dense-only chunks.

In [ ]:
state4 = run_rag("How can I reduce words to their base form in Python NLP?")
print_result(state4)

- **Why visualise:** LangGraph can render the compiled graph as an ASCII diagram or Mermaid chart. This confirms the graph structure matches your intent — all nodes connected correctly, conditional edges showing all branches. When you share this pipeline with a colleague or in documentation, this diagram explains the flow instantly.


In [ ]:
try:
    # Mermaid diagram — works in Jupyter with mermaid extension
    diagram = rag_graph.get_graph().draw_mermaid()
    print("Mermaid diagram:")
    print(diagram)
except Exception:
    pass

# ASCII fallback — always works
print("\nGraph node connections:")
print("  START")
print("    → transform_query")
print("      → retrieve_dense")
print("        → retrieve_sparse")
print("          → fuse_results")
print("            → rerank_chunks")
print("              [confidence >= 0.4]  → generate_answer → END")
print("              [confidence <  0.4 AND retry < 2] → retry_retrieval")
print("                → rerank_chunks (loop)")
print("              [confidence <  0.4 AND retry >= 2] → end_low_confidence → END")
print("              [error] → end_with_error → END")

In [ ]:
# conn.close()

print("=" * 62)
print("NOTEBOOK 11 — LANGGRAPH PIPELINE COMPLETE")
print("=" * 62)
print()
print("  Nodes defined (7 total):")
print("    transform_query    — HyDE query transformation")
print("    retrieve_dense     — pgvector cosine similarity")
print("    retrieve_sparse    — BM25s keyword matching")
print("    fuse_results       — RRF merge + enrichment")
print("    rerank_chunks      — cross-encoder scoring")
print("    generate_answer    — grounded LLM generation")
print("    retry_retrieval    — broadened search on low confidence")
print()
print("  Conditional edge:")
print("    confidence >= 0.4  → generate")
print("    confidence <  0.4  → retry (max 2x)")
print("    retry exhausted    → end_low_confidence")
print("    error in any node  → end_with_error")
print()
print("  Tested on:")
print("    Factual query      → correct answer, no retry")
print("    Exact term query   → correct answer, BM25s contribution")
print("    Out-of-scope query → honest refusal after 2 retries")
print("    Semantic paraphrase→ correct answer via HyDE + dense")
print()
print("  Next step: extract nodes into .py modules")
print("    app/rag/retrieval/  — dense, sparse, fusion, reranker")
print("    app/rag/generation/ — generate_answer")
print("    app/rag/pipeline.py — StateGraph assembly")
print("    app/api/routes.py   — FastAPI endpoint calling rag_graph.invoke()")